In [ ]:
# --- MNIST Digit Classification: Deep Learning with Full Metrics & Curves ---
# Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score,
)
import pandas as pd

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


# MNIST Digit Classification with Deep Learning

**Objective:** Train a feedforward neural network to classify handwritten digits (0–9) from the MNIST dataset. This notebook includes:
- **Training & validation curves** (loss and accuracy over epochs)
- **Cost decrease with backpropagation** (loss per batch within selected epochs)
- **Evaluation metrics:** accuracy, precision, recall, F1-score (per class and macro/weighted), confusion matrix
- **Summary tables** for reporting

---
## 1. Data loading and preprocessing

In [ ]:
# Load MNIST data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values (0-255 → 0-1) for stable gradients in backpropagation
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# One-hot encode the labels for categorical cross-entropy loss
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# Data summary
print("Dataset shapes:")
print(f"  x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"  x_test:  {x_test.shape},  y_test:  {y_test.shape}")
print(f"  Pixel range: [{x_train.min():.2f}, {x_train.max():.2f}]")
print(f"  Classes: {np.unique(y_train)}")


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


---
## 2. Model architecture and training setup

In [ ]:
# Feedforward MLP: Flatten → Dense(128) → Dense(64) → Dense(10, softmax)
model = Sequential([
    Input(shape=(28, 28)),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax"),
])
model.summary()


/opt/anaconda3/envs/tf_env/lib/python3.10/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)


In [ ]:
# Callback to record loss per batch (to visualize cost decrease with backpropagation)
# We record batches for the first 2 epochs only to keep the plot readable.
batch_losses = []  # list of (epoch, batch_idx, loss) for selected epochs
epochs_to_record_batches = {0, 1}  # first two epochs

class BatchLossLogger(Callback):
    def on_batch_end(self, batch, logs=None):
        if logs and "loss" in logs and self.model is not None:
            epoch = getattr(self, "_current_epoch", 0)
            if epoch in epochs_to_record_batches:
                batch_losses.append((epoch, batch, float(logs["loss"])))
    def on_epoch_begin(self, epoch, logs=None):
        self._current_epoch = epoch

batch_callback = BatchLossLogger()

# Training
EPOCHS = 10
BATCH_SIZE = 32
VAL_SPLIT = 0.1

history = model.fit(
    x_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=[batch_callback],
    verbose=1,
)


Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9249 - loss: 0.2538 - val_accuracy: 0.9637 - val_loss: 0.1241
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9666 - loss: 0.1100 - val_accuracy: 0.9745 - val_loss: 0.0864
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9768 - loss: 0.0751 - val_accuracy: 0.9760 - val_loss: 0.0895
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9823 - loss: 0.0568 - val_accuracy: 0.9775 - val_loss: 0.0816
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9857 - loss: 0.0451 - val_accuracy: 0.9793 - val_loss: 0.0777
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9886 - loss: 0.0359 - val_accuracy: 0.9762 - val_loss: 0.0933
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9903 - loss: 0.0299 - val_accuracy: 0.9747 - val_loss: 0.1001
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9913 - loss: 0.0252 - 

In [ ]:
# (Training already done in the cell above; history and batch_losses are available.)


Epoch 1/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9945 - loss: 0.0160 - val_accuracy: 0.9768 - val_loss: 0.1048
Epoch 2/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9938 - loss: 0.0177 - val_accuracy: 0.9785 - val_loss: 0.1044
Epoch 3/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9955 - loss: 0.0129 - val_accuracy: 0.9757 - val_loss: 0.1101
Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9953 - loss: 0.0141 - val_accuracy: 0.9730 - val_loss: 0.1304
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9960 - loss: 0.0121 - val_accuracy: 0.9802 - val_loss: 0.1055
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9958 - loss: 0.0122 - val_accuracy: 0.9787 - val_loss: 0.1187
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9961 - loss: 0.0118 - val_accuracy: 0.9783 - val_loss: 0.1190
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9968 - loss: 0.0096 - 

---
## 3. Training and validation curves

Loss and accuracy over epochs show convergence. **Training loss** decreases as the model fits the data via backpropagation; **validation loss** indicates generalization.

In [ ]:
# Training & validation: Loss and accuracy vs epoch
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(history.history["loss"]) + 1)

# Loss curve
axes[0].plot(epochs_range, history.history["loss"], "b-o", label="Train loss", markersize=4)
axes[0].plot(epochs_range, history.history["val_loss"], "r-s", label="Val loss", markersize=4)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (categorical cross-entropy)")
axes[0].set_title("Cost (loss) over epochs")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(epochs_range, history.history["accuracy"], "b-o", label="Train accuracy", markersize=4)
axes[1].plot(epochs_range, history.history["val_accuracy"], "r-s", label="Val accuracy", markersize=4)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy over epochs")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.1 Cost decrease with backpropagation (per batch)

Within each epoch, the model updates weights after each mini-batch. The plot below shows **loss per batch** for the first two epochs: cost decreases as gradients are computed (backprop) and weights are updated (optimizer step).

In [ ]:
# Batch-level loss (cost decrease with backpropagation)
if batch_losses:
    df_batch = pd.DataFrame(batch_losses, columns=["epoch", "batch", "loss"])
    fig, ax = plt.subplots(figsize=(10, 4))
    for ep in sorted(df_batch["epoch"].unique()):
        sub = df_batch[df_batch["epoch"] == ep]
        ax.plot(sub["batch"], sub["loss"], alpha=0.8, label=f"Epoch {ep + 1}")
    ax.set_xlabel("Batch index (within epoch)")
    ax.set_ylabel("Loss (batch)")
    ax.set_title("Cost decrease with backpropagation (first 2 epochs)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No batch-level losses recorded. Re-run training with BatchLossLogger callback.")

---
## 4. Test set evaluation and metrics

Final performance on the held-out test set: loss, accuracy, and per-class precision, recall, and F1-score.

In [ ]:
# Evaluate on test set (use one-hot labels for loss)
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=1)
y_pred_proba = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

# Summary
print("=" * 50)
print("Test set results")
print("=" * 50)
print(f"  Test loss:     {test_loss:.4f}")
print(f"  Test accuracy: {test_acc:.4f}")
print()

# Per-class and aggregate metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, labels=range(10)
)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro"
)
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted"
)

# Table: per-class metrics
metrics_df = pd.DataFrame({
    "Class": range(10),
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "Support": support.astype(int),
})
total_support = int(support.sum())
metrics_df.loc["Macro avg", :] = ["—", f"{macro_p:.4f}", f"{macro_r:.4f}", f"{macro_f1:.4f}", total_support]
metrics_df.loc["Weighted avg", :] = ["—", f"{weighted_p:.4f}", f"{weighted_r:.4f}", f"{weighted_f1:.4f}", total_support]
print("Per-class and aggregate metrics:")
display(metrics_df)

# Scikit-learn classification report (text)
print("\nClassification report (sklearn):")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))


313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 717us/step - accuracy: 0.9747 - loss: 0.1281
Test accuracy: 0.9747


### 4.1 Confusion matrix

Rows: true label. Columns: predicted label. Diagonal = correct predictions.

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

---
## 5. Summary

Key metrics at a glance (below). Re-run the notebook from the top to reproduce. To change training length or batch size, edit `EPOCHS` and `BATCH_SIZE` in the training cell.

In [ ]:
# Key metrics summary (for reporting)
summary = pd.DataFrame({
    "Metric": [
        "Final train loss",
        "Final validation loss",
        "Test loss",
        "Test accuracy",
        "Macro F1 (test)",
        "Weighted F1 (test)",
    ],
    "Value": [
        f"{history.history['loss'][-1]:.4f}",
        f"{history.history['val_loss'][-1]:.4f}",
        f"{test_loss:.4f}",
        f"{test_acc:.4f}",
        f"{macro_f1:.4f}",
        f"{weighted_f1:.4f}",
    ],
})
display(summary)